# alugaFloripa: Predição de Aluguel Residencial em Florianópolis - Regressão

**Problema preditivo:** estimar o valor mensal de aluguel (`valor`) de um imóvel residencial em Florianópolis a partir de suas características físicas e comerciais. É um problema de **regressão**, pois o alvo é uma variável numérica contínua, em **reais (BRL)**.

**Por que isso importa:** uma estimativa de aluguel confiável pode ajudar proprietários, locatários e imobiliárias a tomarem decisões melhores, evitando preços muito acima ou abaixo do praticado no mercado local.

**Dataset:** `full_history.csv`, base pública de anúncios de imóveis para aluguel em Florianópolis, obtida no Kaggle. Neste projeto, será utilizado o recorte de imóveis com `categoria` igual a **Residencial** e `periodicidade` igual a **Mês**.

## Configuração do ambiente

In [1]:
import os  # Permite interagir com pastas e caminhos do sistema operacional
import sys  # Permite alterar a lista de caminhos que o Python usa para importar módulos
from pathlib import Path  # Ajuda a trabalhar com caminhos de arquivos de forma segura

# Garante que o notebook encontre a raiz do projeto,
# mesmo que ele seja executado dentro da pasta notebooks/
_root = Path.cwd()  # Captura a pasta atual onde o notebook está sendo executado

# Sobe pelas pastas até encontrar a pasta src/
while not (_root / "src").is_dir() and _root != _root.parent:
    _root = _root.parent  # Volta uma pasta acima

os.chdir(_root)  # Define a raiz do projeto como pasta principal de execução

# Adiciona a raiz do projeto ao caminho de importação do Python,
# permitindo importar arquivos da pasta src/
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f"Raiz do projeto configurada em: {_root}")  # Mostra o caminho encontrado

Raiz do projeto configurada em: c:\Users\crisj\OneDrive\Área de Trabalho\SCTECH\Desenvolvimento de IA para Análise Preditiva [T2]\IA\PRojeto Final-M1\alugaFloripa


## Importação das bibliotecas

In [4]:
import pandas as pd  # leitura, organização e análise de dados em tabelas
import numpy as np  # operações numéricas e cálculos com arrays

import matplotlib.pyplot as plt  # criação de gráficos
import seaborn as sns  # criação de gráficos estatísticos

import statsmodels.api as sm  # análises estatísticas auxiliares
from statsmodels.stats.outliers_influence import variance_inflation_factor  # cálculo do VIF para avaliar multicolinearidade

from sklearn.model_selection import train_test_split  # função para separar dados de treino e teste
from sklearn.preprocessing import StandardScaler  # padronizador de variáveis numéricas
from sklearn.linear_model import LinearRegression  # modelo de Regressão Linear
from sklearn.ensemble import RandomForestRegressor  # modelo Random Forest para segunda versão
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # métricas de avaliação de regressão

import joblib  # biblioteca para salvar e carregar modelos treinados
import json  # biblioteca para salvar métricas em arquivo JSON
from datetime import datetime  # registrar data/hora das versões do modelo

# Funções e parâmetros reutilizáveis do projeto, implementados em src/
from src.config import RAW_FILE, PROCESSED_FILE, FINAL_FILE, FIGURES_DIR, MODEL_FILE, METRICS_FILE, TARGET_COL, TEST_SIZE, RANDOM_STATE  # importa caminhos e parâmetros centrais
from src.dataset import load_raw_data, save_processed_data, load_processed_data, save_final_data, load_final_data  # importa funções de leitura e salvamento dos datasets

pd.set_option("display.max_columns", None)  # configura o pandas para mostrar todas as colunas do DataFrame
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")  # formata números com 2 casas decimais

sns.set_theme(style="whitegrid", palette="muted")  # define estilo visual dos gráficos

In [5]:
# Carrega o dataset bruto usando a função de src/dataset.py
df = load_raw_data()

print("INSPEÇÃO INICIAL DO DATASET")
print(f"Shape: {df.shape}")

print(f"\nTipos de dados:\n{df.dtypes}")

print(f"\nValores nulos por coluna:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

df.describe()

INSPEÇÃO INICIAL DO DATASET
Shape: (36092, 24)

Tipos de dados:
Unnamed: 0          int64
data               object
fonte              object
id                  int64
descricao          object
tipo               object
endereco           object
rua                object
numero            float64
bairro             object
cidade             object
valor               int64
periodicidade      object
condominio        float64
area              float64
qtd_banheiros     float64
qtd_quartos       float64
qtd_vagas         float64
url                object
amenities          object
categoria          object
valor_total       float64
valor_m2          float64
valor_condo_m2    float64
dtype: object

Valores nulos por coluna:
rua               9175
numero           18455
qtd_banheiros     2909
qtd_quartos      15125
qtd_vagas        10568
amenities         8491
dtype: int64


,Unnamed: 0,id,numero,valor,condominio,area,qtd_banheiros,qtd_quartos,qtd_vagas,valor_total,valor_m2,valor_condo_m2
count,"36,092.00","36,092.00","17,637.00","36,092.00","36,092.00","36,092.00","33,183.00","20,967.00","25,524.00","36,092.00","36,092.00","36,092.00"
mean,"3,698.61","2,615,194,682.23","1,043.50","7,006.88",308.45,177.01,2.30,2.71,1.98,"7,315.33",48.72,2.91
std,"2,132.01","221,482,894.26","11,950.51","9,275.19",664.49,228.38,1.65,1.37,1.41,"9,441.35",33.34,4.90
min,0.00,"62,568,647.00",1.00,100.00,0.00,4.00,1.00,1.00,1.00,100.00,1.00,0.00
25%,"1,855.00","2,628,922,989.00",140.00,"2,000.00",0.00,50.00,1.00,2.00,1.00,"2,200.00",27.27,0.00
50%,"3,704.00","2,652,437,258.50",371.00,"4,000.00",0.00,90.00,2.00,3.00,2.00,"4,390.00",43.44,0.00
75%,"5,548.00","2,663,872,000.00",950.00,"8,000.00",487.00,210.00,3.00,3.00,2.00,"8,500.00",62.50,6.12
max,"7,446.00","2,674,047,878.00","1,111,111.00","108,000.00","12,645.00","2,000.00",9.00,9.00,9.00,"108,000.00",399.00,35.00
